# 19.14 Machine unlearning

Machine unlearning aims to remove a training point influence without rebuilding the whole system from scratch.

This is a gap topic in the lesson plan, so the notebook states the gap while still building the real influence-style removal and exact retraining comparison. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_wine, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(19)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def fit_scaled_logistic(X, y, test_size=0.4, random_state=0):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf, x_tr, x_te, y_tr, y_te


def fit_scaled_logistic_three_way(X, y, random_state=0):
    if len(y) < 12:
        repeats = int(np.ceil(12 / len(y)))
        X = np.tile(X, (repeats, 1))
        y = np.tile(y, repeats)
    x_tmp, x_te, y_tmp, y_te = train_test_split(X, y, test_size=0.3, random_state=random_state, stratify=y)
    x_tr, x_cal, y_tr, y_cal = train_test_split(x_tmp, y_tmp, test_size=0.35, random_state=random_state, stratify=y_tmp)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_cal = scaler.transform(x_cal)
    x_te = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf, x_tr, x_cal, x_te, y_tr, y_cal, y_te


def top_class_binary_view(y, proba):
    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y).astype(int)
    return correct, conf, pred


def degrade_d5_features(name, X):
    rng = np.random.default_rng(1918)
    if name.startswith("D5"):
        return X + rng.normal(0.0, X.std(axis=0) * 0.35, size=X.shape)
    return X


## The concept, built once

The first-order unlearning approximation is

$$\theta_{unlearn}\approx\theta-H^{-1}\nabla_\theta \ell(z_i)/n.$$

D1 uses ridge logistic-style influence math on a tiny design matrix, and the lesson toy update must recover signed total $0.180$ and absolute mass $0.380$.

In [ ]:

def ridge_fit_closed_form(X, y, lam=1.0):
    design = np.column_stack([np.ones(len(X)), X])
    hessian = design.T @ design + lam * np.eye(design.shape[1])
    theta = np.linalg.solve(hessian, design.T @ y)
    return theta, hessian, design


def unlearn_point(theta, hessian, x_i, y_i, n):
    row = np.r_[1.0, x_i]
    grad_i = row * (row @ theta - y_i)
    update = np.linalg.solve(hessian, grad_i) / n
    theta_unlearned = theta - update
    return theta_unlearned, update

components = np.array([0.240, -0.100, 0.040])
total = float(components.sum())
abs_mass = float(np.abs(components).sum())
share = float(abs(components[0]) / abs_mass)
guarded = total + 0.1 * abs_mass
contrast = total - components[2]
change = contrast - total

assert np.isclose(total, 0.180)
assert np.isclose(abs_mass, 0.380)
assert np.isclose(share, 0.632, atol=0.001)
assert np.isclose(guarded, 0.218)
assert np.isclose(change, -0.040)
print("gap topic: lesson-body numbers are supplied by the rebuild plan")
print("total", total, "abs mass", abs_mass, "share", share)


The reusable method compares approximate removal to exact retraining-without-the-point. Faithfulness is the prediction distance between those two models; smaller is better.

In [ ]:

def unlearning_distance(X, y, delete_fraction=0.02):
    X_small = X[: min(len(X), 180)]
    y_small = (y[: min(len(y), 180)] == np.max(y)).astype(float)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X_small)
    lam = 1.0
    theta, hessian, design = ridge_fit_closed_form(Xs, y_small, lam=lam)
    delete_n = max(1, int(delete_fraction * len(y_small)))
    theta_approx = theta.copy()
    for idx in range(delete_n):
        theta_approx, update = unlearn_point(theta_approx, hessian, Xs[idx], y_small[idx], len(y_small))
    keep = np.ones(len(y_small), dtype=bool)
    keep[:delete_n] = False
    theta_exact, exact_hessian, exact_design = ridge_fit_closed_form(Xs[keep], y_small[keep], lam=lam)
    full_design = np.column_stack([np.ones(len(Xs)), Xs])
    approx_pred = full_design @ theta_approx
    exact_pred = full_design @ theta_exact
    return float(np.mean(np.abs(approx_pred - exact_pred)))


## The dataset ladder

All six notebooks in this batch use the same F15 classification ladder: a hand-checkable D1 toy, synthetic rungs, two real tabular rungs, and the real D5 Breast Cancer stress rung. The method changes, not the data-loading story.

In [ ]:

rungs = clf_ladder()

for name, X, y in rungs:
    classes, counts = np.unique(y, return_counts=True)
    preview = np.round(X[:3, : min(4, X.shape[1])], 3)
    print(name)
    print("  shape:", X.shape)
    print("  classes:", dict(zip(classes.tolist(), counts.tolist())))
    print("  sample columns:")
    print(preview)


In [ ]:

rows = []
for rung, (name, X, y) in enumerate(rungs, start=1):
    distance = unlearning_distance(X, y, delete_fraction=0.02)
    rows.append((rung, name, distance))

print("rung | prediction distance to exact retraining")
for rung, name, distance in rows:
    print(f"D{rung} | {distance:.4f} | {name}")


In [ ]:

fractions = np.array([0.01, 0.02, 0.05, 0.10, 0.20])
summary = []
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for col, (name, X, y) in enumerate(rungs):
    distances = [unlearning_distance(X, y, delete_fraction=f) for f in fractions]
    summary.append(distances[1])
    axes[col].plot(fractions, distances, marker="o")
    axes[col].set_title(name.split("(")[0].strip(), fontsize=9)
    axes[col].set_xlabel("delete fraction")
    axes[col].set_ylabel("distance")

plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), summary, marker="o")
plt.xlabel("ladder rung")
plt.ylabel("distance at 2% delete")
plt.title("Faithfulness vs. D1→D5")
plt.xticks(range(1, 6))
plt.show()


## Pitfall on D5: updating only the main model

A real deletion touches more than parameters. If caches, embeddings, evaluation slices, and index rows are not linked by provenance, the point can survive outside the main model.

In [ ]:

artifacts = {
    "main_model": True,
    "embedding_cache": False,
    "search_index": False,
    "evaluation_slice": False,
}
wrong_remaining = [name for name, removed in artifacts.items() if not removed]

fixed_artifacts = {name: True for name in artifacts}
fixed_remaining = [name for name, removed in fixed_artifacts.items() if not removed]

print("wrong residual artifacts", wrong_remaining)
print("fixed residual artifacts", fixed_remaining)
print("wrong residual count", len(wrong_remaining))
print("fixed residual count", len(fixed_remaining))


## Evaluate it + Practice

- Metric: prediction distance to exact retraining.
- No-skill baseline: do nothing after receiving a deletion request.
- Cheap sanity check: distance should be near zero for a one-point deletion on D1.
- Ablation: delete 20% with the first-order update and compare to exact retraining.
- Failure signals: large deletion batches, missing lineage, or residual cache artifacts.

Practice 1: Change the random seed or stress strength and explain which rung moves most.

Practice 2: Increase the ridge penalty and see whether influence approximation stabilizes.

Practice 3: Add a fake artifact to the provenance checklist and require cleanup.